# Satellite AI Pipeline — Colab Training

**流程：**
1. 環境設置 & GPU 確認
2. Google Drive 掛載 & 專案 clone
3. xView 資料前處理（GeoJSON → YOLO tiles）
4. L1 訓練（YOLOv11m）
5. L1 特徵快取
6. L2 訓練（SatMAE ViT-Base）
7. 推論測試

> **資料擺放（執行前先完成）**
> ```
> MyDrive/satellite-pipeline/
> ├── xview/
> │   ├── train_images/      ← xView 解壓後的影像
> │   └── xView_train.geojson
> └── labels/                ← L2 標籤（military/non-military CSV，見 Cell 9 格式）
> ```

## 1. GPU 確認

In [ ]:
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout)

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## 2. Google Drive 掛載 & 專案設置

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/satellite-pipeline'
os.makedirs(DRIVE_ROOT, exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

In [ ]:
# 專案 clone（第一次執行）或 pull（已存在）
import os

REPO_DIR = '/content/defence-satellite-pipeline'
REPO_URL = 'https://github.com/YOUR_USERNAME/defence-satellite-pipeline.git'  # ← 改成你的 repo URL

if not os.path.exists(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} pull

os.chdir(REPO_DIR)
print(f'Working dir: {os.getcwd()}')

## 3. 安裝套件

In [ ]:
!pip install -q ultralytics>=8.3.0 transformers>=4.35.0 wandb>=0.16.0 rasterio>=1.3.0 scikit-learn>=1.3.0 opencv-python-headless>=4.8.0

# 確認 ultralytics 支援 YOLOv11
from ultralytics import YOLO
import ultralytics
print(f'ultralytics: {ultralytics.__version__}')

In [ ]:
# W&B 登入（可選，不登入仍可訓練，只是沒有遠端追蹤）
import wandb
wandb.login()  # 輸入你的 API key，或按 Enter 跳過

## 4. 路徑設定（集中管理）

In [ ]:
from pathlib import Path

# ── xView 原始資料（從 Google Drive 讀取）──
XVIEW_IMAGE_DIR  = Path(DRIVE_ROOT) / 'xview' / 'train_images'
XVIEW_GEOJSON    = Path(DRIVE_ROOT) / 'xview' / 'xView_train.geojson'

# ── 前處理輸出（本地快速 SSD）──
TILES_DIR        = Path('/content/data/tiles')       # 640×640 tiles
LABELS_DIR       = Path('/content/data/labels_yolo') # YOLO .txt 標籤
DATASET_YAML     = Path('/content/data/xview_dataset.yaml')

# ── L2 標籤（CSV：image_name, label 0/1）──
L2_TRAIN_LABELS  = Path(DRIVE_ROOT) / 'labels' / 'train_labels.csv'
L2_VAL_LABELS    = Path(DRIVE_ROOT) / 'labels' / 'val_labels.csv'

# ── checkpoints（存回 Drive 以跨 session 保存）──
CKPT_DIR         = Path(DRIVE_ROOT) / 'checkpoints'
L1_CKPT          = CKPT_DIR / 'l1' / 'weights' / 'best.pt'
L2_CKPT_DIR      = CKPT_DIR / 'l2'

# ── L1 特徵快取 ──
FEATURES_DIR     = Path('/content/data/l1_features')

for d in [TILES_DIR, LABELS_DIR, FEATURES_DIR, CKPT_DIR, L2_CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print('Paths ready.')
print(f'  xView images : {XVIEW_IMAGE_DIR}')
print(f'  GeoJSON      : {XVIEW_GEOJSON}')
print(f'  Tiles output : {TILES_DIR}')

## 5. 資料前處理 — GeoJSON → YOLO 標籤

In [ ]:
# xView GeoJSON → YOLO .txt 標籤（針對原始大圖）
import sys
sys.path.insert(0, str(REPO_DIR))

from src.data.converters import convert_xview_to_yolo

convert_xview_to_yolo(
    geojson_path=str(XVIEW_GEOJSON),
    image_dir=str(XVIEW_IMAGE_DIR),
    output_dir=str(LABELS_DIR),
)
label_count = len(list(LABELS_DIR.glob('*.txt')))
print(f'Generated {label_count} YOLO label files')

## 6. 資料前處理 — 大圖切 640×640 Tiles

In [ ]:
# 將大圖切成 640×640 tiles，並對應調整 YOLO 標籤座標
# 預計約 30–60 分鐘（CPU-bound）

import json
from PIL import Image
from src.data.tiling import tile_image

TILE_SIZE = 640
OVERLAP   = 64   # 10% overlap 避免邊緣漏偵測
STRIDE    = TILE_SIZE - OVERLAP

TILES_IMG_DIR   = TILES_DIR / 'images'
TILES_LABEL_DIR = TILES_DIR / 'labels'
TILES_IMG_DIR.mkdir(parents=True, exist_ok=True)
TILES_LABEL_DIR.mkdir(parents=True, exist_ok=True)

image_paths = list(XVIEW_IMAGE_DIR.glob('*.tif')) + \
              list(XVIEW_IMAGE_DIR.glob('*.jpg')) + \
              list(XVIEW_IMAGE_DIR.glob('*.png'))
print(f'Found {len(image_paths)} source images')

total_tiles = 0
for idx, img_path in enumerate(image_paths):
    label_src = LABELS_DIR / (img_path.stem + '.txt')
    if not label_src.exists():
        continue  # 無標籤的影像跳過

    tile_infos = tile_image(
        image_path=str(img_path),
        output_dir=str(TILES_IMG_DIR),
        tile_size=TILE_SIZE,
        overlap=OVERLAP,
    )

    # 讀取原圖尺寸以重新計算標籤座標
    with Image.open(img_path) as im:
        orig_w, orig_h = im.size

    orig_labels = []
    for line in label_src.read_text().strip().splitlines():
        parts = line.split()
        cls = int(parts[0])
        cx, cy, bw, bh = float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
        # 轉回像素座標
        orig_labels.append((cls,
            cx * orig_w, cy * orig_h,
            bw * orig_w, bh * orig_h))

    for ti in tile_infos:
        tile_lines = []
        for cls, pcx, pcy, pbw, pbh in orig_labels:
            # bbox in pixel absolute
            x1 = pcx - pbw / 2
            y1 = pcy - pbh / 2
            x2 = pcx + pbw / 2
            y2 = pcy + pbh / 2
            # clip to tile
            tx1 = max(x1, ti.x_offset)
            ty1 = max(y1, ti.y_offset)
            tx2 = min(x2, ti.x_offset + TILE_SIZE)
            ty2 = min(y2, ti.y_offset + TILE_SIZE)
            if tx2 <= tx1 or ty2 <= ty1:
                continue
            # 保留中心點在 tile 內的標籤
            if not (ti.x_offset <= pcx < ti.x_offset + TILE_SIZE and
                    ti.y_offset <= pcy < ti.y_offset + TILE_SIZE):
                continue
            ncx = (pcx - ti.x_offset) / TILE_SIZE
            ncy = (pcy - ti.y_offset) / TILE_SIZE
            nbw = min(pbw, TILE_SIZE) / TILE_SIZE
            nbh = min(pbh, TILE_SIZE) / TILE_SIZE
            tile_lines.append(f'{cls} {ncx:.6f} {ncy:.6f} {nbw:.6f} {nbh:.6f}')

        tile_label = TILES_LABEL_DIR / (Path(ti.tile_path).stem + '.txt')
        tile_label.write_text('\n'.join(tile_lines))
        total_tiles += 1

    if (idx + 1) % 50 == 0:
        print(f'  Processed {idx+1}/{len(image_paths)} images, {total_tiles} tiles so far')

print(f'\nTotal tiles generated: {total_tiles}')

## 7. Train/Val 分割 & dataset YAML 生成

In [ ]:
import random
import shutil
import yaml

random.seed(42)
VAL_RATIO = 0.15

all_tiles = sorted(TILES_IMG_DIR.glob('*.jpg')) + sorted(TILES_IMG_DIR.glob('*.png'))
# 只保留有標籤且標籤非空的 tiles
labeled = [t for t in all_tiles
           if (TILES_LABEL_DIR / (t.stem + '.txt')).exists() and
              (TILES_LABEL_DIR / (t.stem + '.txt')).stat().st_size > 0]
print(f'Tiles with annotations: {len(labeled)} / {len(all_tiles)}')

random.shuffle(labeled)
n_val = int(len(labeled) * VAL_RATIO)
val_tiles   = labeled[:n_val]
train_tiles = labeled[n_val:]

for split, tiles in [('train', train_tiles), ('val', val_tiles)]:
    img_dir   = TILES_DIR / split / 'images'
    label_dir = TILES_DIR / split / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    label_dir.mkdir(parents=True, exist_ok=True)
    for t in tiles:
        shutil.copy(t, img_dir / t.name)
        src_lbl = TILES_LABEL_DIR / (t.stem + '.txt')
        shutil.copy(src_lbl, label_dir / src_lbl.name)

print(f'Train: {len(train_tiles)} | Val: {len(val_tiles)}')

dataset_cfg = {
    'path': str(TILES_DIR),
    'train': 'train/images',
    'val':   'val/images',
    'nc': 4,
    'names': ['aircraft', 'storage_tank', 'vehicle_lot', 'building'],
}
with open(DATASET_YAML, 'w') as f:
    yaml.dump(dataset_cfg, f, default_flow_style=False)
print(f'Dataset YAML written: {DATASET_YAML}')

## 8. L1 訓練（YOLOv11m）

T4 VRAM 16 GB：`batch=32` 通常可跑，視 tile 數量調整。  
> Colab 斷線後從 `resume=True` 繼續，不需重頭。

In [ ]:
import os
os.environ['WANDB_PROJECT'] = 'satellite-l1'

from ultralytics import YOLO

L1_RESUME_CKPT = str(CKPT_DIR / 'l1' / 'weights' / 'last.pt')
resume = os.path.exists(L1_RESUME_CKPT)

if resume:
    print(f'Resuming from {L1_RESUME_CKPT}')
    model = YOLO(L1_RESUME_CKPT)
else:
    print('Starting fresh training with yolo11m.pt')
    model = YOLO('yolo11m.pt')

model.train(
    data=str(DATASET_YAML),
    epochs=100,
    imgsz=640,
    batch=32,          # T4 16GB — 若 OOM 改為 16
    optimizer='AdamW',
    lr0=0.001,
    freeze=10,
    degrees=45.0,
    flipud=0.5,
    mosaic=1.0,
    hsv_h=0.015,
    project=str(CKPT_DIR),
    name='l1',
    save=True,
    resume=resume,
    exist_ok=True,
)
print(f'L1 best checkpoint: {L1_CKPT}')

In [ ]:
# L1 訓練結果摘要
import pandas as pd
results_csv = CKPT_DIR / 'l1' / 'results.csv'
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = df.columns.str.strip()
    print(df[['epoch', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']].tail(10).to_string(index=False))

## 9. L1 特徵快取（L2 訓練前置）

In [ ]:
from src.training.train_l2 import cache_l1_features

# 對 train + val 影像各自快取
for split in ['train', 'val']:
    cache_l1_features(
        image_dir=str(TILES_DIR / split / 'images'),
        l1_checkpoint=str(L1_CKPT),
        output_dir=str(FEATURES_DIR / split),
    )
print('Feature caching complete.')

## 10. L2 標籤格式說明

L2 需要每張 **原始影像**（或 tile 對應的地點）的 military/non-military 標籤。  
CSV 格式（`train_labels.csv` / `val_labels.csv`）：

```
image_name,label
1102.tif,0
1234.tif,1
...
```

- `label=1`：軍事設施
- `label=0`：非軍事

> 若尚未有標籤，先跳過 Cell 11-12，待標籤備妥後再執行。

## 11. L2 訓練（SatMAE ViT-Base）

In [ ]:
import os
os.environ['WANDB_PROJECT'] = 'satellite-l2'

from src.training.train_l2 import train_l2

train_l2(
    config_path='configs/l2_classifier.yaml',
    train_image_dir=str(TILES_DIR / 'train' / 'images'),
    val_image_dir=str(TILES_DIR / 'val' / 'images'),
    features_dir=str(FEATURES_DIR),
    train_labels=str(L2_TRAIN_LABELS),
    val_labels=str(L2_VAL_LABELS),
    checkpoint_dir=str(L2_CKPT_DIR),
    wandb_project='satellite-l2',
)

## 12. 推論測試（單張影像）

In [ ]:
from src.inference.predict import predict
from IPython.display import Image as IPImage, display

TEST_IMAGE = '/content/test_image.jpg'  # ← 換成你要推論的影像路徑

result = predict(
    image_path=TEST_IMAGE,
    l1_checkpoint=str(L1_CKPT),
    l2_checkpoint=str(L2_CKPT_DIR / 'best.pt'),
    output_dir='/content/outputs',
)

print(f"Confidence  : {result['confidence']:.3f}")
print(f"Level       : {result['confidence_level']}")
print(f"Detections  : {result['detection_count']}")

display(IPImage('/content/outputs/annotated.jpg'))

## 13. Checkpoint → Google Drive 備份

In [ ]:
# checkpoints 已直接寫入 DRIVE_ROOT/checkpoints，無需額外複製
print('L1 best.pt :', L1_CKPT, '|', 'exists:', L1_CKPT.exists())
print('L2 best.pt :', L2_CKPT_DIR / 'best.pt', '|', 'exists:', (L2_CKPT_DIR / 'best.pt').exists())